# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane 4 (CTR / Engagement Opportunity Scoring), locked. This notebook builds the transparent,
hand-written baseline that the Week-5 model must beat: two signal checks first, then one rule
with a score, a reason code, an action label, a ranked queue written to disk, and a skeptic's
read of the top of the list.

Everything the rule *uses* comes from March 2026 (`month=2026-03`). April is touched only to
*evaluate* the finished queue — never as an input.

## 0. Setup — same contract slice as ML-04

Same auth pattern and the same page-month contract as w03: one row = one client × page over
March, visible pool = ≥ 100 impressions and positive average position.

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

In [ ]:
for name, src in [('fact_daily month=2026-03', MAR), ('fact_daily month=2026-04', APR)]:
    n, d1, d2 = con.sql(f'SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM {src}').fetchone()
    print(f'{name}: {n:,} rows, {d1} .. {d2}')

## 1. My rule and its reason codes

**The rule in plain words:** *a page earns a review slot if it is visible enough to matter
(≥100 March impressions), sits in a position tier where clicks should be flowing, yet captures
far less CTR than its tier's benchmark.*

**Outputs per page:** `score = 100 × max(0, 1 − capture_ratio) × log₁₀(impressions)` ·
one reason code — `ctr_below_tier_benchmark` · one action label by tier/severity:
`rewrite_title_meta` (page-1 tiers), `improve_intent_match` (positions 11–20),
`monitor_only` (deeper).

The rule leans on two signals. Each gets checked **before** the rule is coded — a bucket table
with n per bucket, and a one-word verdict:

1. **CTR-vs-position** — the signal behind FlyRank's real `needs_ctr_fix` flag
   (`low_ctr_visible_page`: impressions ≥ 500, 0 < position ≤ 20, CTR below threshold).
   If median CTR does not actually fall as position worsens, a position-tier benchmark is
   meaningless and this rule dies here.
2. **Volume** — the signal behind `is_quick_win`. If missed clicks do not concentrate in
   higher-volume pages, weighting by volume chases noise instead of opportunity.

In [ ]:
import numpy as np
import pandas as pd

q_mar = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               AVG(gsc_avg_position) AS pos_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    ),
    climpo AS (
        SELECT client_hash_id, SUM(gsc_impressions) AS cli_imp
        FROM {MAR}
        GROUP BY 1
    )
    SELECT p.*, c.cli_imp
    FROM pagemo p JOIN climpo c USING (client_hash_id)
"""
pool = con.sql(q_mar).df()
pool['ctr_mar'] = pool['clk_mar'] / pool['imp_mar']
print(f'visible March pool: {len(pool):,} page-month rows (imp >= 100, pos > 0)')

### Signal 1 — CTR-vs-position (behind `needs_ctr_fix`)

Observed median CTR by finer position band. The flag logic assumes pages ranking better earn
materially higher CTR — that spread is what makes "expected CTR for your position" a fair bar.

In [ ]:
bands = [(1, 3, 'pos 1-3'), (4, 6, 'pos 4-6'), (7, 10, 'pos 7-10'),
         (11, 15, 'pos 11-15'), (16, 20, 'pos 16-20'), (21, 50, 'pos 21-50'), (51, 10**9, 'pos 51+')]

def band_of(p):
    for lo, hi, name in bands:
        if lo <= p <= hi:
            return name

sig1 = pool.copy()
sig1['band'] = sig1['pos_mar'].apply(band_of)
t1 = (sig1.groupby('band')
      .agg(n=('ctr_mar', 'size'),
           median_ctr=('ctr_mar', 'median'),
           p25=('ctr_mar', lambda s: s.quantile(.25)),
           p75=('ctr_mar', lambda s: s.quantile(.75)))
      .reindex([b[2] for b in bands]).round(4))
display(t1)

meds = t1['median_ctr'].tolist()
drops = sum(1 for a, b in zip(meds, meds[1:]) if b < a)
verdict1 = ('CONFIRMED' if drops == len(meds) - 1
            else 'OPPOSITE' if drops == 0
            else 'MIXED')
print(f'verdict: {verdict1}  ({drops} of {len(meds)-1} adjacent steps show falling median CTR)')
print(f'spread: pos 1-3 median {meds[0]:.3f}% vs pos 51+ median {meds[-1]:.4f}%')

### Signal 2 — Volume (behind `is_quick_win`)

The quick-win logic assumes absolute opportunity concentrates where impressions are: a small
relative gap on a high-volume page is worth more clicks than the same gap on a quiet one.
Missed clicks here = Σ (tier benchmark − actual CTR) × impressions over under-capturing pages.

In [ ]:
def tier_of(pos):
    if pos <= 3:
        return 'p1_top'
    if pos <= 10:
        return 'p1'
    if pos <= 20:
        return 'p2'
    return 'deep'

chk = pool.copy()
chk['tier'] = chk['pos_mar'].apply(tier_of)
bench_chk = chk[chk['imp_mar'] >= 1000].groupby('tier')['ctr_mar'].median()
chk['bench'] = chk['tier'].map(bench_chk)
chk['missed'] = ((chk['bench'] - chk['ctr_mar']).clip(lower=0)) * chk['imp_mar']

imp_bands = [(100, 249, 'imp 100-249'), (250, 499, 'imp 250-499'), (500, 999, 'imp 500-999'),
             (1000, 4999, 'imp 1k-5k'), (5000, 10**12, 'imp 5k+')]

def imp_band(i):
    for lo, hi, name in imp_bands:
        if lo <= i <= hi:
            return name

chk['imp_band'] = chk['imp_mar'].apply(imp_band)
cap = chk.assign(cap_ratio=chk['ctr_mar'] / chk['bench']).groupby('imp_band')['cap_ratio'].median()
t2 = chk.groupby('imp_band').agg(n=('missed', 'size'), total_missed_clicks=('missed', 'sum'))
t2['median_capture_ratio'] = cap.round(3)
t2['share_of_missed_clicks'] = (t2['total_missed_clicks'] / t2['total_missed_clicks'].sum()).round(3)
t2['share_of_pages'] = (t2['n'] / t2['n'].sum()).round(3)
t2 = t2.reindex([b[2] for b in imp_bands])
t2['total_missed_clicks'] = t2['total_missed_clicks'].round(0)
display(t2)

top2_missed = t2['share_of_missed_clicks'].iloc[-2:].sum()
top2_pages = t2['share_of_pages'].iloc[-2:].sum()
verdict2 = ('CONFIRMED' if top2_missed > top2_pages + 0.05
            else 'OPPOSITE' if top2_missed < top2_pages - 0.05
            else 'MIXED')
print(f'verdict: {verdict2}  (top two volume bands hold {top2_missed:.0%} of missed clicks '
      f'but only {top2_pages:.0%} of pages)')

## 2. Build the ranked queue (writes the CSV)

Coded exactly as announced — no fitted weights, nothing from April. Pages whose tier benchmark
is missing or zero (no reliable bar to compare against) are excluded from the queue, with the
count reported. The CSV is written **before** April is ever joined — the queue structurally
cannot contain outcome information.

In [ ]:
rule = pool.copy()
rule['tier'] = rule['pos_mar'].apply(tier_of)
bench_counts = rule[rule['imp_mar'] >= 1000].groupby('tier').size().rename('bench_support')
bench = (rule[rule['imp_mar'] >= 1000].groupby('tier')['ctr_mar']
         .median().rename('tier_expected_ctr'))
rule['tier_expected_ctr'] = rule['tier'].map(bench)
rule['bench_support'] = rule['tier'].map(bench_counts)

before = len(rule)
rule = rule[rule['tier_expected_ctr'] > 0].copy()
print(f'queue pool: {len(rule):,} rows ({before - len(rule)} dropped: no/zero tier benchmark)')

rule['capture_ratio'] = rule['ctr_mar'] / rule['tier_expected_ctr']
rule['score'] = (100 * (1 - rule['capture_ratio']).clip(lower=0) * np.log10(rule['imp_mar'])).round(1)
rule['reason_code'] = 'ctr_below_tier_benchmark'

action_by_tier = {'p1_top': 'rewrite_title_meta', 'p1': 'rewrite_title_meta',
                  'p2': 'improve_intent_match', 'deep': 'monitor_only'}
rule['action_label'] = rule['tier'].map(action_by_tier)

rule = rule.sort_values('score', ascending=False).reset_index(drop=True)
rule.insert(0, 'rank', np.arange(1, len(rule) + 1))

import pathlib
out_dir = pathlib.Path('work/outputs')
out_dir.mkdir(parents=True, exist_ok=True)
cols = ['rank', 'client_hash_id', 'content_hash_id', 'tier', 'pos_mar',
        'imp_mar', 'clk_mar', 'ctr_mar', 'tier_expected_ctr',
        'capture_ratio', 'score', 'reason_code', 'action_label']
rule[cols].to_csv(out_dir / 'baseline_action_score.csv', index=False)
print(f'wrote work/outputs/baseline_action_score.csv ({len(rule):,} rows)')
rule[cols].head(10)

In [ ]:
# Evaluation ONLY from here: April joins after the CSV exists, never before.
q_apr = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_apr,
           SUM(gsc_clicks)      AS clk_apr
    FROM {APR}
    GROUP BY 1, 2
"""
apr = con.sql(q_apr).df()

ev = rule.merge(apr, on=['client_hash_id', 'content_hash_id'], how='inner').copy()
ev['apr_ctr'] = ev['clk_apr'] / ev['imp_apr']
bench_apr = ev[ev['imp_apr'] >= 1000].groupby('tier')['apr_ctr'].median().rename('tier_expected_apr')
ev['tier_expected_apr'] = ev['tier'].map(bench_apr)

labeled = ev[(ev['imp_apr'] >= 100) & (ev['tier_expected_ctr'] > 0) &
             (ev['tier_expected_apr'] > 0)].copy()
labeled['under_captured_apr'] = ((labeled['apr_ctr'] / labeled['tier_expected_apr']) < 0.5).astype(int)

from sklearn.dummy import DummyClassifier

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:k]].mean())

base_rate = float(labeled['under_captured_apr'].mean())
dummy = DummyClassifier(strategy='most_frequent').fit(
    labeled[['score']], labeled['under_captured_apr'])
dummy_p50 = p_at_k(dummy.predict_proba(labeled[['score']])[:, 1],
                   labeled['under_captured_apr'], 50)

p50 = p_at_k(labeled['score'], labeled['under_captured_apr'], 50)
p20 = p_at_k(labeled['score'], labeled['under_captured_apr'], 20)

print(f'labeled evaluation frame : {len(labeled):,} rows (April-present subset of {len(rule):,})')
print(f'label base rate          : {base_rate:.3f}')
print(f'dummy floor (P@50)       : {dummy_p50:.3f}')
print(f'RULE Precision@20        : {p20:.3f}')
print(f'RULE Precision@50        : {p50:.3f}')

import json, datetime
metrics = {
    'notebook': 'w04_baseline_score.ipynb',
    'generated_at_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
    'lane': 'L4_ctr_engagement_opportunity',
    'feature_month': '2026-03',
    'outcome_month_eval_only': '2026-04',
    'pool_rows': int(len(rule)),
    'labeled_rows': int(len(labeled)),
    'label_base_rate': round(base_rate, 4),
    'dummy_precision_at_50': round(dummy_p50, 4),
    'rule_precision_at_20': round(p20, 4),
    'rule_precision_at_50': round(p50, 4),
    'signal_verdicts': {'ctr_vs_position_behind_needs_ctr_fix': verdict1,
                        'volume_behind_is_quick_win': verdict2},
    'score_formula': '100 * max(0, 1 - ctr/tier_expected_ctr) * log10(impressions)',
    'reason_code': 'ctr_below_tier_benchmark',
}
with open('work/outputs/w04_baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('wrote work/outputs/w04_baseline_metrics.json (committed receipts)')

## 3. Top-20 review

For each pick: the action, why it sits here, and what would make it wrong. The
"what-would-make-it-wrong" column is generated from four concrete failure modes — thin
benchmark support, position near a tier boundary, low volume, modest gap — because those are
the honest ways a rule like this misfires.

In [ ]:
top = labeled.head(20).copy()

def why(r):
    return (f"captures {r['capture_ratio']:.0%} of its tier benchmark "
            f"({r['ctr_mar']:.2f}% vs {r['tier_expected_ctr']:.2f}%) at avg position {r['pos_mar']:.1f}, "
            f"{int(r['imp_mar']):,} impressions")

def wrong(r):
    reasons = []
    if r['bench_support'] < 30:
        reasons.append(f"benchmark rests on only {int(r['bench_support'])} high-volume pages")
    near_boundary = min(abs(r['pos_mar'] - 3), abs(r['pos_mar'] - 10), abs(r['pos_mar'] - 20)) <= 0.5
    if near_boundary:
        reasons.append('position sits on a tier boundary; a slightly different band flips its benchmark')
    if r['imp_mar'] < 200:
        reasons.append('low volume: the CTR estimate itself is noisy')
    if r['capture_ratio'] > 0.8:
        reasons.append('gap is modest - may be normal variation, not a fixable flaw')
    if not reasons:
        reasons.append('CTR gap may reflect search intent that metadata edits cannot fix')
    return '; '.join(reasons)

top['why_here'] = top.apply(why, axis=1)
top['what_would_make_it_wrong'] = top.apply(wrong, axis=1)
top['content_short'] = top['content_hash_id'].str[:10]
top['client_short'] = top['client_hash_id'].str[:10]

show = top[['rank', 'content_short', 'client_short', 'tier', 'pos_mar', 'imp_mar',
            'score', 'action_label', 'why_here', 'what_would_make_it_wrong']]
with pd.option_context('display.max_colwidth', 60, 'display.width', 220):
    display(show)

conc = top['client_short'].value_counts()
print(f'client concentration in top 20: busiest client holds {conc.iloc[0]} of 20 slots')
print(f"action mix: {top['action_label'].value_counts().to_dict()}")

### Skeptic's read of the top 20

*(filled after execution — what the table above actually shows, which picks survive scrutiny
and which do not, and whether the reason code is doing real work or rubber-stamping every row.)*

## 4. Weak picks + leakage check

Which flagged picks look least trustworthy, and proof that nothing illegal entered the score:
no product flags, no April columns, no fitted weights — the score is computed and written to
disk before any outcome data joins the frame.

In [ ]:
weak = top[top['what_would_make_it_wrong'].str.contains('benchmark rests|boundary|low volume')]
print(f'clearly weak picks among the top 20: {len(weak)}')
if len(weak):
    display(weak[['rank', 'content_short', 'what_would_make_it_wrong']])

april_cols_in_rule = [c for c in ['imp_apr', 'clk_apr', 'apr_ctr', 'tier_expected_apr', 'under_captured_apr']
                      if c in rule.columns]
product_flags_used = [c for c in ['health_score', 'needs_ctr_fix', 'is_quick_win', 'priority_score']
                      if c in rule.columns]
checks = {
    'queue built before April merge': True,
    'no April column in rule frame': len(april_cols_in_rule) == 0,
    'no product flags used as input': len(product_flags_used) == 0,
    'single fixed reason code': rule['reason_code'].nunique() == 1,
    'no fitted weights (formula only)': True,
}
print()
for k, v in checks.items():
    print(f'  [{"x" if v else " "}] {k}: {"OK" if v else "FAIL"}')
assert all(checks.values()), 'leakage check failed'
print('leakage check passed')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.